# Week 1 - Foundations: The SVD and Eigen-Images for Recognition

One decomposition runs through almost this entire course: the **singular value
decomposition** (SVD). Hand it a data matrix and it returns the directions along which
the data actually varies, ordered by how much variation each one carries. Week 1 meets
the SVD on images. On a library of **real** peripheral-blood-cell crops (BloodMNIST, via
the course data layer) the SVD's directions are **eigen-cells** -- shared image-space
patterns that every cell is a weighted sum of -- and we use them to compress, reconstruct,
and classify. The famous version of this idea is *eigenfaces*; here it is eigen-cells.

The **singular values** are the thread of the week, because they do two jobs at once. First,
they rank the eigen-cells by how much library variance each explains, so keeping the top few and
dropping the small-$\sigma$ tail is **compression**. Second, they measure **conditioning**: when
you *solve* a linear system in this basis you divide by singular values, and dividing by a tiny
one turns a small error in the data into a large error in the answer. The same small-$\sigma$ tail
is therefore both the low-variance part *and*, in a solve, the numerically dangerous part -- so
keeping the top modes is one move that serves representation, compression, and numerical stability
at once. We open with a compact linear-system solve to make "dividing by a tiny singular value
amplifies error" concrete (that is *conditioning*), then carry the same singular values into the
eigen-cells.

The course's discipline is already here: never trust a number you haven't checked. Every step is
quantified -- the conditioning of the solve, the variance a few modes capture, reconstruction
error versus basis size, and classifier accuracy on data it never saw during fitting.

**Reading.** Kutz, *Data-Driven Modeling & Scientific Computation*, 2nd ed., in the order we use
them: **Chapter 2, sections 1--5** for the linear-system warm-up ($A\mathbf{x}=\mathbf{b}$ by
direct and iterative solvers, steepest descent, eigenvalues and solvability, and
eigenvalue/eigenvector face recognition), then **Chapter 15, sections 1--5** for the core -- the
singular value decomposition, principal component analysis, and proper orthogonal modes, the
machinery behind our eigen-images. Read them for the derivations; the treatment below is in our
own terms and runs against our own fixtures and the BloodMNIST library.

**Learning goals** (in the order the lesson builds them).

- Solve a linear system $A\mathbf{x}=\mathbf{b}$ and use the condition number
  $\kappa = \sigma_{\max}/\sigma_{\min}$ to judge how far the answer can be trusted -- seeing
  first-hand how small singular values amplify error.
- Build an eigen-image basis by mean-centering an image library and taking its principal
  components with the singular value decomposition (`numpy.linalg.svd`), and read the spectrum
  as explained-variance ratios -- how many modes reach 90/95/99% of the variance.
- Relate the number of modes kept to reconstruction error, and see why truncating to the top
  modes -- dropping the small-singular-value tail -- is at once compression and the conditioning
  control from Section 1.
- Classify images in the compact eigen-basis with a nearest-neighbour rule on the official
  held-out test partition, and read the result honestly -- a compact PCA basis *preserves* the
  recognition signal rather than sharpening it -- stating its limitations plainly.

```{admonition} Which paradigm?
:class: note
**Data-driven.** You never write down a model of what a blood cell should look like. You mean-center the BloodMNIST microscopy crops, let the SVD hand you the eigen-cells -- the axes along which the images actually vary -- then reconstruct and classify in that learned basis with a nearest-neighbour rule. Those coordinates come inductively from the pixels, not from cell biology or the microscope's optics. That puts this week at the data-driven extreme of the course; next week makes the opposite, mechanistic move, committing to a dose-response model and fitting its parameters.
```


## Setup

We seed every random number generator and apply the course plotting style, so the numbers below
are reproducible from a cold kernel.

In [ ]:
# Colab setup: install the ddm4bio course library.
# No-op when ddm4bio is already importable (e.g. the course-site build), so
# this cell is safe everywhere. It is hidden from the rendered site via the
# "remove-cell" tag, but runs when this notebook is opened in Google Colab.
try:
    import ddm4bio  # noqa: F401
except ModuleNotFoundError:
    %pip install -q "ddm4bio @ git+https://github.com/symbiont-ai/ddm4bio.git"
    import ddm4bio  # noqa: F401

In [ ]:
import numpy as np

import ddm4bio
from ddm4bio import seed_everything
from ddm4bio.viz.style import set_style

seed_everything()
set_style()

print(f"ddm4bio version: {ddm4bio.__version__}")

## 1. Warm-up: singular values decide when a solve can be trusted

Before the images, a short look at *why* singular values matter -- on the simplest problem that
has them, a linear system $A\mathbf{x}=\mathbf{b}$. Write $A$ through its singular value
decomposition, $A = U\Sigma V^\top$. Solving then divides each piece of the answer by a singular
value,

$$\mathbf{x} = \sum_i \frac{\mathbf{u}_i^\top \mathbf{b}}{\sigma_i}\,\mathbf{v}_i,$$

so the moment a singular value $\sigma_i$ is tiny, that division magnifies any error in
$\mathbf{b}$ into a large error in $\mathbf{x}$. The ratio of largest to smallest singular value,
$\kappa = \sigma_{\max}/\sigma_{\min}$, is the **condition number**, and it bounds that
magnification.

We make it concrete with two symmetric positive-definite systems of the same size but very
different singular-value spreads -- one well-conditioned, one badly so -- each with a *known* true
solution. (For an SPD matrix the singular values are the eigenvalues, so we build each matrix
straight from a chosen spectrum -- here running geometrically from $1$ down to a genuinely tiny
$1/\kappa$, so the smallest singular value really is small.) We then solve each **through the SVD
itself** -- compute $U\Sigma V^\top$ and form
$\mathbf{x} = \sum_i (\mathbf{u}_i^\top\mathbf{b}/\sigma_i)\,\mathbf{v}_i$ -- so the division by each
$\sigma_i$ is right there in the code. We measure the **residual** $\|A\hat{\mathbf{x}}-\mathbf{b}\|$
(how well the answer satisfies the equation), the **forward error**
$\|\hat{\mathbf{x}}-\mathbf{x}_{\text{true}}\|$ (how close it is to the truth), and -- because we
solved mode by mode -- *where* that error lives across the singular values.

In [ ]:
def spd_with_cond(n, cond, seed):
    """A symmetric positive-definite matrix built from a chosen singular-value spectrum.

    For an SPD matrix the singular values equal the eigenvalues, so we set them directly --
    geometrically spaced from 1 down to ``1 / cond`` -- so the largest singular value is 1, the
    smallest is a genuinely tiny ``1 / cond``, and the condition number is exactly kappa = cond.
    """
    q, _ = np.linalg.qr(np.random.default_rng(seed).standard_normal((n, n)))
    singular_values = np.geomspace(1.0, 1.0 / cond, n)
    return (q * singular_values) @ q.T


def probe(A, x_true, noise_rel=1e-8):
    """Solve A x = b THROUGH THE SVD with a slightly noisy b, and measure everything."""
    b = A @ x_true
    noise = np.random.default_rng(1).standard_normal(A.shape[0])
    noise *= noise_rel * np.linalg.norm(b) / np.linalg.norm(noise)   # small, measurement-like
    b_noisy = b + noise

    U, s, vt = np.linalg.svd(A)                       # the SVD, computed on A itself
    x_hat = vt.T @ ((U.T @ b_noisy) / s)              # x = sum_i (u_i^T b / sigma_i) v_i

    err_by_mode = np.abs(U.T @ noise) / s             # |u_i^T noise| / sigma_i: the error, mode by mode
    k_tail = max(1, s.size // 10)                     # the smallest-sigma decile of modes
    forward = np.linalg.norm(x_hat - x_true) / np.linalg.norm(x_true)
    rel_in = np.linalg.norm(noise) / np.linalg.norm(b)
    return {
        "kappa": s[0] / s[-1],                        # condition number, straight from the singular values
        "residual": np.linalg.norm(A @ x_hat - b_noisy) / np.linalg.norm(b_noisy),
        "forward": forward,
        "rel_in": rel_in,
        "amplification": forward / rel_in,
        "tail_share": float((err_by_mode[-k_tail:] ** 2).sum() / (err_by_mode ** 2).sum()),
    }


n = 100
x_true = np.random.default_rng(0).standard_normal(n)
A_well = spd_with_cond(n, cond=8.0, seed=42)      # well-conditioned SPD
A_ill = spd_with_cond(n, cond=1e8, seed=43)       # ill-conditioned SPD
print(f"Built two {n}x{n} SPD systems from chosen singular-value spectra "
      f"(kappa = 8 and 1e8), each with a known true solution.")

We now solve each system through the SVD with a *slightly noisy* right-hand side -- the kind of
small measurement or round-off error real data always carries -- and read off the key numbers.
Two are easy to confuse: the **residual** measures how well our answer satisfies the equation we
actually solved, while the **forward error** measures how far that answer is from the true
solution. And because we solved mode by mode, we can also see *where* the error collects. Watch
the residual stay at machine precision while the forward error -- and the share of it living in
the smallest-$\sigma$ modes -- blows up as $\kappa$ grows.

In [ ]:
for label, A in [("Well-conditioned", A_well), ("Ill-conditioned", A_ill)]:
    r = probe(A, x_true)
    print(f"\n{label}  (kappa = {r['kappa']:.1e}, read off the singular values):")
    print(f"  relative residual  ||A x_hat - b|| / ||b||        : {r['residual']:.1e}")
    print(f"  forward error      ||x_hat - x_true|| / ||x_true|| : {r['forward']:.1e}")
    print(f"  input noise {r['rel_in']:.1e} -> output error {r['forward']:.1e}"
          f"  (amplification {r['amplification']:.1e}x, bounded by kappa)")
    print(f"  share of that error in the smallest-sigma 10% of modes : {r['tail_share']:.0%}")

**QC note.** Both systems were "solved" to a relative residual at machine precision -- the computed
answer satisfies its (noisy) equation about as well as floating point allows. Yet on the
well-conditioned system the forward error is tiny too, while on the ill-conditioned one it is
enormous: a solution that nails the equation to ~15 digits is still *wrong in the second decimal
place*. **A small residual does not imply an accurate solution.** The SVD shows exactly why:
writing $\mathbf{x} = \sum_i (\mathbf{u}_i^\top\mathbf{b}/\sigma_i)\,\mathbf{v}_i$, the smallest
singular values divide the noise by almost nothing, so the error piles into the small-$\sigma$
modes -- and the total amplification is bounded by $\kappa = \sigma_{\max}/\sigma_{\min}$. Hold
onto this: the small-$\sigma$ modes are the untrustworthy ones -- and they are the same low-variance
tail we *drop* (rather than divide by) when we truncate the eigen-basis in a moment.

## 2. From pixels to an image library

The same tool -- the SVD -- now meets a *library* of images. Here the library is **real**: we pull
BloodMNIST -- peripheral-blood-cell microscopy crops from MedMNIST v2 -- through the course data
layer, `get_dataset("bloodmnist")`. Each crop is a small RGB image carrying an integer cell-type
label. We convert every crop to grayscale (averaging the colour channels) and flatten it to a
vector, using the **whole** training partition to build the basis (and the whole official test
partition to score). The singular values of this library will play the very two roles we just
saw -- ranking directions by variance, and telling us which ones to trust.

In [ ]:
from ddm4bio.datasets import get_dataset

ds = get_dataset("bloodmnist", seed=0)
print(f"Data source : {ds.source}")
print(f"Provenance  : {ds.provenance}")

# BloodMNIST ships an OFFICIAL train / validation / test split (a 7:1:2 partition), so we use
# it rather than re-splitting. We use the WHOLE training partition to build the eigen-basis and
# the classifier, and score on the WHOLE official test partition -- nothing is subsampled. The
# colour channels are averaged to grayscale.
def to_grayscale(imgs):
    gray = imgs.mean(axis=-1)                              # (n, H, W) grayscale
    return gray, gray.reshape(gray.shape[0], -1).astype(float)


images, X = to_grayscale(ds.payload["train_images"])
y = ds.payload["train_labels"].ravel()
_, X_test = to_grayscale(ds.payload["test_images"])
y_test = ds.payload["test_labels"].ravel()
img_h, img_w = images.shape[1], images.shape[2]
classes = np.unique(np.concatenate([y, y_test]))
n_classes = classes.size

print(f"Training library : {X.shape[0]} images of {img_h}x{img_w} pixels ({n_classes} cell classes)")
print(f"Official test set: {X_test.shape[0]} held-out images (used only to score)")
print(f"Flattened feature matrix X: {X.shape} (samples x pixels)")

# BloodMNIST publishes named cell types (MedMNIST v2; Acevedo et al., 2020). We label the
# real crops with these; the offline fallback is synthetic, so it keeps generic class ids.
BLOODMNIST_CLASSES = {
    0: "basophil", 1: "eosinophil", 2: "erythroblast", 3: "immature granulocyte",
    4: "lymphocyte", 5: "monocyte", 6: "neutrophil", 7: "platelet",
}


def class_label(i):
    return BLOODMNIST_CLASSES[int(i)] if ds.source == "real" else f"class {int(i)}"

A quick look at the cell types the recognizer has to work with -- one representative crop per
class (the *medoid*: the crop whose pixels are closest to its class mean, so it is the most
typical example rather than an arbitrary one). Even these representatives are low-resolution,
variable, and noisy.

In [ ]:
import matplotlib.pyplot as plt

# One representative crop per cell type: the class medoid -- the crop whose pixels are closest
# to that class's mean image, so it is the most typical example rather than an arbitrary one.
ncols = 4
nrows = int(np.ceil(n_classes / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.0 * ncols, 2.3 * nrows))
for i, ax in enumerate(np.atleast_1d(axes).ravel()):
    if i < n_classes:
        idx = np.where(y == classes[i])[0]
        if idx.size:
            medoid = idx[np.argmin(np.linalg.norm(X[idx] - X[idx].mean(axis=0), axis=1))]
            ax.imshow(images[medoid], cmap="gray_r")
            ax.set_title(class_label(classes[i]), fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"One representative crop per cell type ({n_classes} classes, {img_h}x{img_w} pixels)")
fig;

Before modeling, one look at the **class balance** of the training library. The eight cell
types are not equally represented -- worth keeping in mind when we read the 1-NN confusion
matrix later, since a nearest-neighbour rule leans toward the crowded classes and struggles
on the rare ones.

In [ ]:
train_classes, train_counts = np.unique(y, return_counts=True)
order = np.argsort(train_counts)[::-1]                     # most common first
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(range(len(order)), train_counts[order], color='0.6', edgecolor='white')
ax.bar_label(bars, padding=2, fontsize=8)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([class_label(train_classes[i]) for i in order],
                   rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Training images')
ax.set_title(f'Class balance of the {y.size}-image training library')
ax.margins(y=0.12)
fig;

## 3. Building the eigen-image basis (the ground-truth-adjacent check)

The eigen-image recipe is exactly PCA on the image library:

1. **Mean-center** -- subtract the average image so the basis describes
   *deviations* from the mean, not the mean itself.
2. **Take principal components** -- the right singular vectors of the centered
   data matrix are the eigen-images: orthogonal full-resolution pixel patterns (one weight per
   image pixel), ordered by how much library variance each explains.
3. **Project** -- every image becomes a short vector of coordinates in this
   basis (its PCA scores).

Because PCA is a *deterministic* linear algebra operation (an SVD), its "ground
truth" is self-checking: the explained-variance ratios are guaranteed to be
non-negative and to sum to one, and the top modes must reconstruct the data
better than any other orthogonal basis of the same size. We verify those
invariants explicitly rather than take them on faith.

In [ ]:
# Mean image and centered library.
mean_image = X.mean(axis=0)
X_centered = X - mean_image

# The eigen-images ARE the principal components: the right singular vectors of the centered
# library. NumPy's SVD returns them directly, ordered by how much variance each explains.
_, singular_values, vt = np.linalg.svd(X_centered, full_matrices=False)
evr = singular_values**2 / np.sum(singular_values**2)   # explained-variance ratio per mode

# How many independent directions can the data hold? The rank of a mean-centered matrix is
# capped by BOTH counts: rank <= min(n_samples - 1, n_pixels) -- centering costs one degree
# of freedom. With many more images than pixels the library is PIXEL-limited (rank = pixels);
# with fewer images than pixels it is SAMPLE-limited (rank = n_samples - 1, and that last
# centering direction is a numerically-zero artifact). We count the modes that carry signal
# with the tolerance rule np.linalg.matrix_rank uses, from the singular values already in hand.
rank_tol = singular_values[0] * max(X_centered.shape) * np.finfo(singular_values.dtype).eps
effective_rank = int(np.sum(singular_values > rank_tol))

print(f"Explained-variance ratios sum to 1: {np.isclose(evr.sum(), 1.0)}")
print(f"All ratios non-negative and non-increasing: "
      f"{np.all(evr >= 0) and np.all(np.diff(evr) <= 1e-12)}")
print(f"Variance captured by mode 1 alone : {evr[0]:.1%}")
print(f"Variance captured by top 10 modes : {evr[:10].sum():.1%}")
_n, _p = X_centered.shape
_regime = "pixel-limited" if _n - 1 >= _p else "sample-limited (centering kills one)"
print(f"Effective rank (modes above tol)  : {effective_rank} = min(n-1, pixels) = min({_n - 1}, {_p})  [{_regime}]")

The scree curve shows how fast the explained variance decays. Unlike the sharp
rank-2 elbow of a purely synthetic fixture, real image data has a *gentle*
shoulder: a handful of modes dominate, but a long tail of small modes carries
finer, lower-variance variation. What that tail *contains* is not settled by the
spectrum alone -- some may be class-relevant morphology, some may be nuisance
variation (illumination, staining, segmentation) or noise; the plot tells us only
that each of these modes carries little variance.

In [ ]:
from ddm4bio.viz.plots import scree_plot

ax = scree_plot(evr[:20])       # first 20 modes; the tail is a slow decay to zero
ax.set_title("Scree plot: explained variance of the top 20 eigen-images")
ax.figure;

Every reconstruction in this section starts from one picture: the **mean cell**, the
pixel-wise average of the training library. PCA writes each image as this mean *plus* a
weighted sum of the eigen-images that follow, so it is worth seeing on its own first.

In [ ]:
fig, ax = plt.subplots(figsize=(3.3, 3.3))
im = ax.imshow(mean_image.reshape(img_h, img_w), cmap='gray_r')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Mean cell (library average)')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='mean intensity')
fig;

Now the eigen-images themselves -- the *signed directions of variation* in image
space, not template or prototype cells. Each is a full-resolution pixel pattern
reshaped back to the image grid. The first few look like smooth blobs that capture
gross cell shape and brightness; later ones encode progressively finer,
higher-frequency contrasts. Any library image is a weighted sum of the mean image
plus these patterns. One caveat on reading them: each mode's overall sign is an
arbitrary SVD convention -- flip a pattern's light and dark together and flip its
coordinate for every image, and nothing observable changes -- and it can differ
across linear-algebra backends. Treat the *structure* of each eigen-image as
meaningful, not its polarity.

In [ ]:
n_show = 8
eigen_images = vt[:n_show]                     # top-n signed variation patterns (rows of Vt)
vmax = float(np.abs(eigen_images).max())       # ONE scale, symmetric about zero, for all panels
ncols = 4
nrows = int(np.ceil(n_show / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(2.1 * ncols, 2.35 * nrows),
                         constrained_layout=True)
for i, ax in enumerate(axes.ravel()):
    if i < n_show:
        im = ax.imshow(eigen_images[i].reshape(img_h, img_w),
                       cmap='RdBu_r', vmin=-vmax, vmax=vmax)   # symmetric -> 0 maps to white
        ax.set_title(f'PC {i + 1}   ({evr[i]:.1%})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Top 8 eigen-images ('eigen-cells'), each labelled with its explained variance")
fig.colorbar(im, ax=axes, shrink=0.85, fraction=0.05, pad=0.02,
             label='signed loading  (blue < 0 < red)')
fig;

## 4. Reconstruction error vs. number of modes

How many eigen-images do we actually need? Project each image onto the top $k$ modes,
reconstruct it, and measure the error. As $k$ grows the reconstruction tightens; the useful
question is where the curve flattens -- the point past which extra modes buy little fidelity.
Keeping the top $k$ and discarding the rest drops exactly the small-$\sigma$ tail Section 1
flagged: it carries little variance *and* the smallest, least-trustworthy singular values, so
letting it go costs almost no fidelity and quietly suppresses noise (and, were we solving rather
than reconstructing, it is the truncation that tames the ill-conditioning). We report a single
**global (Frobenius) relative error** over the whole library,
$\lVert X_c - \hat{X}_c\rVert_F / \lVert X_c\rVert_F$ -- one number for the entire
sample-by-pixel matrix, *not* an average of per-image errors -- and overlay the
variance-captured milestones (90/95/99%). Because it is the global Frobenius error, it obeys an
*exact* identity with the variance spectrum, which we check below; we then look at how the error
is distributed across individual images.

In [ ]:
def reconstruct_with_k(X_centered, vt, k):
    """Project onto the top-k eigen-images and map back to pixel space."""
    basis = vt[:k]                      # (k, n_pixels)
    scores = X_centered @ basis.T       # (n_samples, k)
    return scores @ basis               # (n_samples, n_pixels), centered reconstruction


def global_frobenius_error(reference, approx):
    """One relative error for the WHOLE sample-by-pixel matrix: ||ref - approx||_F / ||ref||_F.

    This is a global (energy-weighted) Frobenius error, not the average of per-image relative
    errors -- high-norm images count for more. The next cell looks at the per-image spread.
    """
    return float(np.linalg.norm(reference - approx) / np.linalg.norm(reference))


# Variance milestones first, so the mode sweep and its plot span the real basis.
cum_evr = np.cumsum(evr)


def modes_for(threshold):
    return int(np.searchsorted(cum_evr, threshold) + 1)


k90, k95, k99 = modes_for(0.90), modes_for(0.95), modes_for(0.99)
n_modes = effective_rank            # the modes that carry signal (Section 3)

# A sweep that spans the whole basis -- adapting to the real 28x28 data (hundreds of modes)
# and the 8x8 offline fallback alike -- and always includes the variance milestones.
base = [1, 2, 4, 8, 16, 32, 64, 128, 256]
k_values = sorted({k for k in base if k < n_modes} | {k90, k95, k99, n_modes})
errors = [global_frobenius_error(X_centered, reconstruct_with_k(X_centered, vt, k))
          for k in k_values]

# Self-check: for the GLOBAL Frobenius error, reconstruction error is not merely correlated
# with explained variance -- it equals the square root of the leftover variance exactly,
#     ||X_c - X_c^(k)||_F / ||X_c||_F  ==  ||s[k:]|| / ||s||.
# We form the right-hand side from the TAIL of the singular values. (The tempting form
# sqrt(1 - cum_evr[k-1]) is algebraically identical but loses every digit to catastrophic
# cancellation near full rank -- 1 minus a sum that already equals ~1 -- the very trap
# Section 1 warned about, so we avoid it here.)
s_norm = np.linalg.norm(singular_values)
tail_error = [float(np.linalg.norm(singular_values[k:]) / s_norm) for k in k_values]
assert np.allclose(errors, tail_error, rtol=0, atol=1e-12), \
    "reconstruction error must equal sqrt(leftover variance)"

print(f"Modes to reach 90% variance: {k90}")
print(f"Modes to reach 95% variance: {k95}")
print(f"Modes to reach 99% variance: {k99}  (of {n_modes} effective modes)")
print("Self-check passed: global Frobenius error == sqrt(leftover variance) for every k.")
for k, e in zip(k_values, errors):
    print(f"  k={k:3d}:  global Frobenius reconstruction error = {e:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, errors, marker="o", linewidth=1.5)
ax.set_xscale("log")
for (k, name), y_lab in zip([(k90, "90%"), (k95, "95%"), (k99, "99%")], [0.62, 0.40, 0.62]):
    ax.axvline(k, color="0.6", linestyle="--", linewidth=1)
    ax.text(k, y_lab, f"{name}\n(k={k})", fontsize=8, color="0.35", ha="center",
            backgroundcolor="white")
ax.set_xlabel("Number of eigen-images (k, log scale)")
ax.set_ylabel("Global (Frobenius) reconstruction error")
ax.set_title("Reconstruction error falls as the eigen-basis grows")
fig;

That curve is one number per $k$ for the entire library -- but the global Frobenius
error hides real spread across images. At the 95% milestone some crops reconstruct far
better than others. The histogram below is the *per-image* relative error at $k=k_{95}$:
the distribution the single global number summarizes.

In [ ]:
k_dist = k95
recon_k = reconstruct_with_k(X_centered, vt, k_dist)
per_image_err = (np.linalg.norm(X_centered - recon_k, axis=1)
                 / np.linalg.norm(X_centered, axis=1))
global_err = global_frobenius_error(X_centered, recon_k)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(per_image_err, bins=30, color="0.6", edgecolor="white")
ax.axvline(global_err, color="C3", linestyle="--", linewidth=1.5,
           label=f"global Frobenius = {global_err:.3f}")
ax.axvline(per_image_err.mean(), color="C0", linestyle=":", linewidth=1.5,
           label=f"mean per-image = {per_image_err.mean():.3f}")
ax.set_xlabel(f"Per-image relative L2 reconstruction error at k={k_dist}")
ax.set_ylabel("Number of images")
ax.set_title(f"Reconstruction quality varies across images (k={k_dist}, 95% variance)")
ax.legend(fontsize=8)
fig;
print(f"Per-image relative error at k={k_dist}: "
      f"median {np.median(per_image_err):.3f}, "
      f"90th pct {np.quantile(per_image_err, 0.9):.3f}, "
      f"worst {per_image_err.max():.3f}  "
      f"(global Frobenius {global_err:.3f}, mean per-image {per_image_err.mean():.3f})")

A visual confirmation: the same cell crop reconstructed from an increasing
number of modes. With only a few eigen-images it is a smudge; it takes on the
order of a hundred modes to sharpen on this real library, and past the 99%
variance milestone the extra modes change little.

In [ ]:
sample_idx = 0
ks_to_show = sorted({1, 8, k95, k99, n_modes})
fig, axes = plt.subplots(1, len(ks_to_show) + 1, figsize=(2.0 * (len(ks_to_show) + 1), 2.2))
axes[0].imshow(images[sample_idx], cmap="gray_r")
axes[0].set_title("original", fontsize=9)
axes[0].set_xticks([]); axes[0].set_yticks([])
for ax, k in zip(axes[1:], ks_to_show):
    recon = reconstruct_with_k(X_centered, vt, k)[sample_idx] + mean_image
    ax.imshow(recon.reshape(img_h, img_w), cmap="gray_r")
    ax.set_title(f"k={k}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"Reconstructing one blood-cell crop ({class_label(y[sample_idx])}) from k eigen-images")
fig;

## 5. Recognition on the official held-out test set

The payoff: classification in the compact basis, scored honestly. BloodMNIST ships an
official train / test split, so we use it rather than re-splitting the training images: we
learn the eigen-basis, the mean image, and the mode count from the **training** partition
alone, then classify each image of the **official test** partition by its nearest training
neighbour. No preprocessing, basis, mode count, or classifier ever sees the test set.

One caveat: this is an honest held-out estimate, not a "leakage-free" one. We drop the few
test crops that are pixel-identical to training ones, but with no donor identifiers we cannot
rule out same-donor near-duplicates across the split.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Guard the official boundary: DROP any test crop byte-identical to a training crop, so the
# held-out score is clean of the exact-duplicate leakage we can detect. (Same-donor NEAR-
# duplicates we cannot detect -- BloodMNIST ships no donor ids -- the interpretation says so.)
train_rows = {row.tobytes() for row in X}
keep = np.array([row.tobytes() not in train_rows for row in X_test])
n_dup = int((~keep).sum())
X_test, y_test = X_test[keep], y_test[keep]
print(f"Training library: {X.shape[0]} images   Official test set: {keep.size} images")
print(f"Removed {n_dup} test crops byte-identical to a training crop; scoring on {X_test.shape[0]}.")

# The eigen-basis, mean, and mode count were all fit on the TRAINING partition (Sections
# 3-4); k is 95% of the training-library variance, chosen without ever touching the test set.
k_class = k95
Z_train = (X - mean_image) @ vt[:k_class].T
Z_test = (X_test - mean_image) @ vt[:k_class].T   # TRAIN mean & basis applied to the test set
print(f"Classifying in a {k_class}-dimensional eigen-basis (down from {X.shape[1]} raw pixels).")

In [ ]:
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(Z_train, y)
acc_eigen = knn.score(Z_test, y_test)

# Baseline: the same 1-NN rule on raw pixels, for an honest comparison.
knn_raw = KNeighborsClassifier(n_neighbors=1)
knn_raw.fit(X, y)
acc_raw = knn_raw.score(X_test, y_test)

print(f"1-NN accuracy in the {k_class}-mode eigen-basis : {acc_eigen:.3f}")
print(f"1-NN accuracy on raw {X.shape[1]} pixels (baseline)   : {acc_raw:.3f}")
print(f"Dimensionality reduction: {X.shape[1]} -> {k_class} "
      f"({100 * k_class / X.shape[1]:.0f}% of the features), "
      f"accuracy change {acc_eigen - acc_raw:+.3f}")

The eigen-basis accuracy essentially matches the raw-pixel baseline -- and that is the
*point*, not a disappointment. PCA chooses its axes to capture **variance**, which is not
the same as **class separation**: the leading modes describe how the images vary, not how
the cell types differ, so a compact PCA basis *preserves* the classification signal rather
than sharpening it. A big accuracy jump here would actually be suspicious. When the goal is
classification *and* labels are available, the right move is to project onto
class-discriminating axes instead -- linear discriminant analysis -- which the course takes
up later. Here PCA earns its keep by **compression**, not by beating raw pixels.

The confusion matrix shows *where* the recognizer struggles. Rather than assume a pattern,
we print the largest off-diagonal cells below and read them directly: any clustering among
cell types with similar *grayscale* morphology is a hypothesis to check against the matrix,
not a given -- and recall we discarded colour and staining cues when we converted to
grayscale, so the recognizer cannot be confusing cells on those. This class-resolved view is
exactly what a single accuracy number hides.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, knn.predict(Z_test), labels=classes)
fig, ax = plt.subplots(figsize=(6.8, 5.8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted cell type")
ax.set_ylabel("True cell type")
ax.set_title(f"1-NN confusion matrix in the eigen-basis (k={k_class})")
ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
ax.set_xticklabels([class_label(c) for c in classes], rotation=45, ha="right", fontsize=7)
ax.set_yticklabels([class_label(c) for c in classes], fontsize=7)
for i in range(n_classes):
    for j in range(n_classes):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    fontsize=7, color="0.2" if cm[i, j] < cm.max() / 2 else "white")
fig.colorbar(im, ax=ax, fraction=0.046, label="count")
fig;

# Show the largest off-diagonal confusions so the "where it struggles" reading is
# demonstrated, not asserted.
off = cm.copy()
np.fill_diagonal(off, 0)
flat = np.argsort(off.ravel())[::-1]
print("Largest confusions (true -> predicted : count):")
for f in flat[:5]:
    i, j = np.unravel_index(f, off.shape)
    if off[i, j] == 0:
        break
    print(f"  {class_label(classes[i])} -> {class_label(classes[j])} : {off[i, j]}")

**Why the held-out test set matters.** We fit the eigen-basis, the mean image, the number of
modes, and the classifier using only the training partition, then evaluated on the official
test partition, which none of that fitting had touched. The reported accuracy is therefore an
estimate of performance on *new* images, not a memorization score. Fitting the representation on the
full (pooled train-and-test) data can bias the test estimate, often optimistically, because
information from the test distribution enters the fitted pipeline -- though on any single
finite split a contaminated analysis can move either way.

In [ ]:
from ddm4bio.interpret import interpretation_block, show_interpretation

acc_se = float(np.sqrt(acc_eigen * (1.0 - acc_eigen) / y_test.size))
acc_lo, acc_hi = acc_eigen - 1.96 * acc_se, acc_eigen + 1.96 * acc_se

# Scoped to THIS section's claim (the classifier): one plain sentence plus the caveats that
# bear on it, folded into the limitations. The scree-tail, storage, and compression points
# live in Sections 3-4 where they were made -- not re-dumped here.
block = interpretation_block(
    claim=(
        f"In a {k_class}-mode eigen-basis ({100 * k_class / X.shape[1]:.0f}% of the pixels), a "
        f"1-nearest-neighbour classifier matches raw-pixel accuracy on the held-out test set "
        f"({acc_eigen:.2f} vs {acc_raw:.2f}): the compact representation keeps the recognition "
        "signal rather than sharpening or losing it."
    ),
    limitations_list=[
        f"The held-out estimate (95% CI [{acc_lo:.2f}, {acc_hi:.2f}], n={y_test.size}) may be "
        f"mildly optimistic: we removed {n_dup} test crops byte-identical to training ones, but "
        "with no donor identifiers, same-donor near-duplicates across the split can't be excluded.",
        "Grayscale only -- we discarded the colour and staining cues that real blood-cell "
        "typing relies on.",
        "1-NN is a deliberately simple recognizer chosen for transparency, not a strong one.",
    ],
)
show_interpretation(block)

## Exercises

Your graded work for this week is **Problem Set 1 (PS1) -- "The Eigen-Subspace as a
Model of Normal Cells"**, distributed and auto-graded through GitHub Classroom. It keeps
this lesson's eigen-image basis but turns it to a new question: if the top principal axes
capture what a *normal* cell looks like, what does the **residual** -- the part that does not
fit -- tell you? The basis primitives (`eigen_basis`, `project`, `reconstruct`) and the data
loader are provided; you fill in the analysis on real BloodMNIST.

- **Part A -- denoise by low-rank projection.** A clean image lives in a few principal axes
  while additive noise spreads across all of them, so projecting a noisy crop onto the top-*k*
  subspace and reconstructing keeps the signal and discards most of the noise -- but only at
  the right rank. Implement `snr_db`, `denoise`, and `best_rank_for_denoising`, and explain
  why the SNR-versus-rank curve rises, peaks, and falls.
- **Part B -- flag out-of-QC images by reconstruction error.** An image that does not belong
  to the normal subspace reconstructs badly, so its reconstruction error is a novelty score
  for catching corrupted acquisitions that add out-of-subspace structure (sensor noise,
  saturation/clipping, debris). Implement
  `reconstruction_anomaly_score`, `detection_auc` (the detector's ROC-AUC), and
  `flag_threshold` (a false-alarm-bounded cutoff), and close with an interpretation block.

Refer to the [PS1 repository README](https://github.com/symbiont-ai/ddm4bio/tree/main/problem_sets/ps1_eigen_recognition) for the submission and auto-grading details.